In [ ]:
pip install rarfile

In [ ]:
import rarfile
import os

# The source_path variable already contains the path to the rar file from the kernel state.
source_path = '/content/data(1).rar'

# Create an output directory for the extracted files if it doesn't exist
extraction_directory = '/content/extracted_rar_files'
os.makedirs(extraction_directory, exist_ok=True)

print(f"Extracting '{source_path}' to '{extraction_directory}'...")

try:
    with rarfile.RarFile(source_path, 'r') as rf:
        rf.extractall(path=extraction_directory)
    print(f"Successfully extracted all files from '{source_path}' to '{extraction_directory}'.")

    # List the contents of the extracted directory
    print("\nContents of the extracted directory:")
    for root, dirs, files in os.walk(extraction_directory):
        for name in files:
            print(os.path.join(root, name))
        for name in dirs:
            print(os.path.join(root, name) + '/')

except rarfile.BadRarFile:
    print(f"Error: '{source_path}' is not a valid RAR file or is corrupted.")
except FileNotFoundError:
    print(f"Error: The file '{source_path}' was not found.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Extracting '/content/data(1).rar' to '/content/extracted_rar_files'...
Successfully extracted all files from '/content/data(1).rar' to '/content/extracted_rar_files'.

Contents of the extracted directory:
/content/extracted_rar_files/2023-11/
/content/extracted_rar_files/2023-10/
/content/extracted_rar_files/2024-05/
/content/extracted_rar_files/2024-08/
/content/extracted_rar_files/2023-12/
/content/extracted_rar_files/2024-06/
/content/extracted_rar_files/2024-11/
/content/extracted_rar_files/2024-09/
/content/extracted_rar_files/2024-10/
/content/extracted_rar_files/2024-04/
/content/extracted_rar_files/2024-01/
/content/extracted_rar_files/2024-02/
/content/extracted_rar_files/2024-07/
/content/extracted_rar_files/2024-03/
/content/extracted_rar_files/2023-11/2023-11-02_05-30-00.yaml
/content/extracted_rar_files/2023-11/2023-11-17_05-30-00.yaml
/content/extracted_rar_files/2023-11/2023-11-12_05-30-00.yaml
/content/extracted_rar_files/2023-11/2023-11-03_05-30-00.yaml
/content/extrac

In [ ]:
import os
import yaml
import pandas as pd

# Directory where RAR files were extracted
extraction_directory = '/content/extracted_rar_files'

all_data = []
yaml_files_found = 0

print(f"Searching for YAML files in '{extraction_directory}'...")

# Walk through the extracted directory to find all YAML files
for root, dirs, files in os.walk(extraction_directory):
    for file in files:
        if file.endswith('.yaml'):
            file_path = os.path.join(root, file)
            yaml_files_found += 1
            try:
                with open(file_path, 'r') as f:
                    data = yaml.safe_load(f)
                    if isinstance(data, dict):
                        all_data.append(data)
                    elif isinstance(data, list):
                        all_data.extend(data)
                    else:
                        print(f"Skipping {file_path}: unexpected YAML format (not dict or list).")
            except yaml.YAMLError as e:
                print(f"Error reading YAML file {file_path}: {e}")
            except Exception as e:
                print(f"An unexpected error occurred while processing {file_path}: {e}")

print(f"Found and processed {yaml_files_found} YAML files.")

if all_data:
    # Create a DataFrame from the collected data
    df = pd.DataFrame(all_data)

    # Define the output CSV file path
    output_csv_path = '/content/combined_data.csv'

    # Save the DataFrame to a CSV file
    df.to_csv(output_csv_path, index=False)

    print(f"Successfully combined all YAML data into '{output_csv_path}'.")
    print("First 5 rows of the combined DataFrame:")
    print(df.head())
    print(f"Shape of the combined DataFrame: {df.shape}")
else:
    print("No valid YAML data found to create a CSV file.")


Searching for YAML files in '/content/extracted_rar_files'...
Found and processed 284 YAML files.
Successfully combined all YAML data into '/content/combined_data.csv'.
First 5 rows of the combined DataFrame:
       Ticker    close                 date     high      low    month  \
0        SBIN   572.10  2023-11-02 05:30:00   575.45   567.60  2023-11   
1  BAJFINANCE  7449.10  2023-11-02 05:30:00  7615.00  7411.20  2023-11   
2       TITAN  3201.50  2023-11-02 05:30:00  3228.50  3180.00  2023-11   
3         ITC   431.15  2023-11-02 05:30:00   432.70   428.65  2023-11   
4         TCS  3360.10  2023-11-02 05:30:00  3375.95  3343.20  2023-11   

      open    volume  
0   571.10  11148516  
1  7530.00    855030  
2  3200.00    774180  
3   430.05   8152031  
4  3343.20   1245384  
Shape of the combined DataFrame: (14200, 8)


In [ ]:
import os
import pandas as pd

# Load the combined data CSV
combined_csv_path = '/content/combined_data.csv'
try:
    df = pd.read_csv(combined_csv_path)
    print(f"Successfully loaded '{combined_csv_path}'. Shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: '{combined_csv_path}' not found.")
    # Exit if the file is not found
    exit()

# Create a directory to store the ticker-based CSVs
output_dir = '/content/ticker_csv_files'
os.makedirs(output_dir, exist_ok=True)
print(f"Created output directory: '{output_dir}'")

# Get unique tickers
unique_tickers = df['Ticker'].unique()
print(f"Found {len(unique_tickers)} unique tickers.")

# Iterate through each ticker and save its data to a separate CSV
for ticker in unique_tickers:
    ticker_df = df[df['Ticker'] == ticker]
    output_file_path = os.path.join(output_dir, f'{ticker}.csv')
    ticker_df.to_csv(output_file_path, index=False)
    print(f"Saved data for ticker '{ticker}' to '{output_file_path}'. Shape: {ticker_df.shape}")

print(f"Successfully converted the combined data into {len(unique_tickers)} separate CSV files in '{output_dir}'.")


Successfully loaded '/content/combined_data.csv'. Shape: (14200, 8)
Created output directory: '/content/ticker_csv_files'
Found 50 unique tickers.
Saved data for ticker 'SBIN' to '/content/ticker_csv_files/SBIN.csv'. Shape: (284, 8)
Saved data for ticker 'BAJFINANCE' to '/content/ticker_csv_files/BAJFINANCE.csv'. Shape: (284, 8)
Saved data for ticker 'TITAN' to '/content/ticker_csv_files/TITAN.csv'. Shape: (284, 8)
Saved data for ticker 'ITC' to '/content/ticker_csv_files/ITC.csv'. Shape: (284, 8)
Saved data for ticker 'TCS' to '/content/ticker_csv_files/TCS.csv'. Shape: (284, 8)
Saved data for ticker 'LT' to '/content/ticker_csv_files/LT.csv'. Shape: (284, 8)
Saved data for ticker 'TATACONSUM' to '/content/ticker_csv_files/TATACONSUM.csv'. Shape: (284, 8)
Saved data for ticker 'RELIANCE' to '/content/ticker_csv_files/RELIANCE.csv'. Shape: (284, 8)
Saved data for ticker 'HCLTECH' to '/content/ticker_csv_files/HCLTECH.csv'. Shape: (284, 8)
Saved data for ticker 'JSWSTEEL' to '/content/t

In [ ]:
import os
import pandas as pd

output_dir = '/content/ticker_csv_files'

print(f"Displaying contents of CSV files in '{output_dir}':")

# Check if the directory exists
if not os.path.exists(output_dir):
    print(f"Error: Directory '{output_dir}' not found.")
else:
    csv_files = [f for f in os.listdir(output_dir) if f.endswith('.csv')]

    if not csv_files:
        print("No CSV files found in the directory.")
    else:
        for file_name in sorted(csv_files):
            file_path = os.path.join(output_dir, file_name)
            try:
                df_ticker = pd.read_csv(file_path)
                print(f"\n--- File: {file_name} ---")
                print(f"Shape: {df_ticker.shape}")
                print("First 5 rows:")
                print(df_ticker.head().to_markdown(index=False))
            except Exception as e:
                print(f"Error reading {file_name}: {e}")

Displaying contents of CSV files in '/content/ticker_csv_files':

--- File: ADANIENT.csv ---
Shape: (284, 8)
First 5 rows:
| Ticker   |   close | date                |    high |     low | month   |    open |   volume |
|:---------|--------:|:--------------------|--------:|--------:|:--------|--------:|---------:|
| ADANIENT | 2215.3  | 2023-11-02 05:30:00 | 2297.95 | 2204.2  | 2023-11 | 2247    |  2340997 |
| ADANIENT | 2208.8  | 2023-11-17 05:30:00 | 2237.95 | 2201    | 2023-11 | 2205.9  |   916518 |
| ADANIENT | 2211.35 | 2023-11-12 05:30:00 | 2226    | 2205.1  | 2023-11 | 2219.7  |   148320 |
| ADANIENT | 2229.85 | 2023-11-03 05:30:00 | 2279    | 2215    | 2023-11 | 2215.3  |  1853300 |
| ADANIENT | 2358.55 | 2023-11-30 05:30:00 | 2409    | 2343.05 | 2023-11 | 2400.05 |  2983879 |

--- File: ADANIPORTS.csv ---
Shape: (284, 8)
First 5 rows:
| Ticker     |   close | date                |   high |    low | month   |   open |   volume |
|:-----------|--------:|:--------------------|----

In [ ]:
import pandas as pd
import os

# Assuming your dataframe is named 'df'
# Create a folder to store the output files if it doesn't exist
output_dir = '/content/ticker_csv_files'
os.makedirs(output_dir, exist_ok=True)

# Group by the 'Ticker' column and save each group
for ticker, data in df.groupby('Ticker'):
    # Clean ticker name for filename (remove special characters if necessary)
    filename = f"{ticker}.csv"
    filepath = os.path.join(output_dir, filename)

    # Save to CSV (index=False avoids adding the row numbers to the file)
    data.to_csv(filepath, index=False)
    print(f"Saved: {filepath}")

Saved: /content/ticker_csv_files/ADANIENT.csv
Saved: /content/ticker_csv_files/ADANIPORTS.csv
Saved: /content/ticker_csv_files/APOLLOHOSP.csv
Saved: /content/ticker_csv_files/ASIANPAINT.csv
Saved: /content/ticker_csv_files/AXISBANK.csv
Saved: /content/ticker_csv_files/BAJAJ-AUTO.csv
Saved: /content/ticker_csv_files/BAJAJFINSV.csv
Saved: /content/ticker_csv_files/BAJFINANCE.csv
Saved: /content/ticker_csv_files/BEL.csv
Saved: /content/ticker_csv_files/BHARTIARTL.csv
Saved: /content/ticker_csv_files/BPCL.csv
Saved: /content/ticker_csv_files/BRITANNIA.csv
Saved: /content/ticker_csv_files/CIPLA.csv
Saved: /content/ticker_csv_files/COALINDIA.csv
Saved: /content/ticker_csv_files/DRREDDY.csv
Saved: /content/ticker_csv_files/EICHERMOT.csv
Saved: /content/ticker_csv_files/GRASIM.csv
Saved: /content/ticker_csv_files/HCLTECH.csv
Saved: /content/ticker_csv_files/HDFCBANK.csv
Saved: /content/ticker_csv_files/HDFCLIFE.csv
Saved: /content/ticker_csv_files/HEROMOTOCO.csv
Saved: /content/ticker_csv_file

In [ ]:
import pandas as pd

def get_top_green_stocks(df):
    # 1. Ensure date is in datetime format
    df['date'] = pd.to_datetime(df['date'])

    # 2. Sort by Ticker and Date to ensure chronological order
    df = df.sort_values(['Ticker', 'date'])

    # 3. Group by Ticker to get the first (opening) and last (closing) price
    summary = df.groupby('Ticker').agg(
        start_price=('close', 'first'),
        end_price=('close', 'last')
    )

    # 4. Calculate Percentage Return
    summary['return_pct'] = ((summary['end_price'] - summary['start_price']) / summary['start_price']) * 100

    # 5. Sort by return and take the top 10
    top_10 = summary.sort_values(by='return_pct', ascending=False).head(10)

    return top_10

# Usage:
# Assuming 'df' is your existing 14200-row DataFrame
top_10_performers = get_top_green_stocks(df)
print(top_10_performers)

            start_price  end_price  return_pct
Ticker                                        
TRENT           2059.10    6652.80  223.092613
BEL              139.20     280.85  101.760057
M&M             1537.40    3012.95   95.976974
BAJAJ-AUTO      5016.45    9481.65   89.011153
BHARTIARTL       925.30    1569.30   69.599049
POWERGRID        199.55     336.95   68.854924
BPCL             170.68     285.85   67.477150
HEROMOTOCO      3015.60    4794.10   58.976655
SUNPHARMA       1141.45    1795.30   57.282404
HCLTECH         1238.70    1898.40   53.257447


In [ ]:
%pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 38.3 MB/s eta 0:00:00


In [ ]:
import mysql.connector
from mysql.connector import Error

def create_db_connection():
    """Establishes a connection to the local MySQL server."""
    try:
        connection = mysql.connector.connect(
            host = "gateway01.ap-southeast-1.prod.aws.tidbcloud.com",
            port = 4000,
            user = "NJeTyJuU9wAnHkN.root",
            password = "kQqBTCcbZqOWb12p" # starting database
        )

        if connection.is_connected():
            print("Successfully connected to the database")
            return connection

    except Error as e:
        print(f"Error while connecting to MySQL: {e}")
        return None

# Usage
conn = create_db_connection()

# Remember to close the connection when finished
if conn and conn.is_connected():
    conn.close()
    print("MySQL connection is closed")

Successfully connected to the database
MySQL connection is closed
